# Emoji Prediction – Three Models (TF‑IDF, Baseline Transformer, FGM‑Adversarial Fine‑Tuning)

This notebook implements:
1. **TF‑IDF + Logistic Regression** – lightweight baseline.
2. **Baseline Transformer** – pretrained RoBERTa‑emoji, evaluated without fine‑tuning.
3. **FGM‑Adversarially Fine‑Tuned Transformer** – hyperparameter tuning (≤3 combos, stratified 20% subsample), class‑weighted loss, early stopping (patience=4), best model saved by validation macro F1, and final full training.

**Device fallback:** 2x GPU → 1x GPU → CPU.

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import warnings
warnings.filterwarnings('ignore')

os.environ["TOKENIZERS_PARALLELISM"] = "false"
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = secret_value_0

if torch.cuda.is_available():
    device_count = torch.cuda.device_count()
    if device_count >= 2:
        device = torch.device("cuda")
        print(f"Using {device_count} GPUs with DataParallel")
    else:
        device = torch.device("cuda")
        print("Using single GPU")
else:
    device = torch.device("cpu")
    print("Using CPU")

train_df = pd.read_csv("/kaggle/input/datasets/nidalshahin/epft-processed-data/EPFT_train.csv")
val_df   = pd.read_csv("/kaggle/input/datasets/nidalshahin/epft-processed-data/EPFT_val.csv")
test_df  = pd.read_csv("/kaggle/input/datasets/nidalshahin/epft-processed-data/EPFT_test.csv")

unique_labels = sorted(train_df['label'].unique())
label_to_id = {lbl: i for i, lbl in enumerate(unique_labels)}
id_to_label = {i: lbl for lbl, i in label_to_id.items()}
num_classes = len(unique_labels)

for df in [train_df, val_df, test_df]:
    df['label_id'] = df['label'].map(label_to_id)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}, Classes: {num_classes}")

class_weights = compute_class_weight('balanced', classes=np.unique(train_df['label_id']), y=train_df['label_id'])
class_weights = torch.tensor(np.clip(class_weights, 0.5, 5.0), dtype=torch.float).to(device)

Using 2 GPUs with DataParallel
Train: 58498, Val: 4922, Test: 2520, Classes: 11


## 1. Model 1: TF‑IDF + Logistic Regression

In [2]:
tfidf_vec = TfidfVectorizer(max_features=15000, ngram_range=(1,3), stop_words='english')
X_train_tfidf = tfidf_vec.fit_transform(train_df['text'])
X_val_tfidf   = tfidf_vec.transform(val_df['text'])
X_test_tfidf  = tfidf_vec.transform(test_df['text'])

lr = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, multi_class='ovr')
lr.fit(X_train_tfidf, train_df['label_id'])

y_pred_test = lr.predict(X_test_tfidf)
test_acc_lr = accuracy_score(test_df['label_id'], y_pred_test)
test_f1_lr  = f1_score(test_df['label_id'], y_pred_test, average='macro')
print(f"TF‑IDF + LR  → Test Acc: {test_acc_lr:.4f}, Macro F1: {test_f1_lr:.4f}")

TF‑IDF + LR  → Test Acc: 0.4171, Macro F1: 0.3327


## 2. Model 2: Baseline Transformer (no fine‑tuning)

In [3]:
tokenizer = AutoTokenizer.from_pretrained("cardiffnlp/twitter-roberta-base-emoji-latest")
model_base = AutoModelForSequenceClassification.from_pretrained(
    "cardiffnlp/twitter-roberta-base-emoji-latest",
    num_labels=num_classes,
    ignore_mismatched_sizes=True
).to(device)

class EmojiDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].flatten(),
            'attention_mask': enc['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

test_dataset = EmojiDataset(test_df['text'].tolist(), test_df['label_id'].tolist(), tokenizer)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

model_base.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        logits = model_base(input_ids, attention_mask=attention_mask).logits
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc_base = accuracy_score(all_labels, all_preds)
test_f1_base  = f1_score(all_labels, all_preds, average='macro')
print(f"Baseline Transformer → Test Acc: {test_acc_base:.4f}, Macro F1: {test_f1_base:.4f}")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-emoji-latest
Key                        | Status   |                                                                                        
---------------------------+----------+----------------------------------------------------------------------------------------
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([100]) vs model:torch.Size([11])          
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([100, 768]) vs model:torch.Size([11, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Baseline Transformer → Test Acc: 0.0710, Macro F1: 0.0458


## 3. Model 3: FGM‑Adversarially Fine‑Tuned Transformer

### 3.1 FGM Implementation

In [4]:
class FGM:
    def __init__(self, model, epsilon=0.5):
        self.model = model
        self.epsilon = epsilon
        self.backup = {}
    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and 'embedding' in name:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isnan(norm):
                    r_at = self.epsilon * param.grad / norm
                    param.data.add_(r_at)
    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and 'embedding' in name and name in self.backup:
                param.data = self.backup[name]
        self.backup.clear()

### 3.2 Helper: Training & Evaluation Functions

In [5]:
def evaluate(model, dataloader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            logits = model(input_ids, attention_mask=attention_mask).logits
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return all_preds, all_labels

def compute_metrics(preds, labels):
    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average='macro')
    return acc, f1_macro

def train_one_epoch(model, train_loader, optimizer, criterion, device, fgm=None, scaler=None):
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss.mean()
        
        if scaler:
            scaler.scale(loss).backward()
        else:
            loss.backward()
        
        if fgm:
            fgm.attack()
            outputs_adv = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss_adv = outputs_adv.loss.mean()
            if scaler:
                scaler.scale(loss_adv).backward()
            else:
                loss_adv.backward()
            fgm.restore()
        
        if scaler:
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()
        optimizer.zero_grad()
        
        total_loss += loss.item()
    return total_loss / len(train_loader)

### 3.3 Hyperparameter Tuning (≤3 combos, stratified 20% subsample)

In [6]:
full_train_val = pd.concat([train_df, val_df], ignore_index=True)
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
_, tune_idx = next(splitter.split(full_train_val['text'], full_train_val['label_id']))
tune_df = full_train_val.iloc[tune_idx].reset_index(drop=True)

tune_split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
tune_train_idx, tune_val_idx = next(tune_split.split(tune_df['text'], tune_df['label_id']))
tune_train = tune_df.iloc[tune_train_idx].reset_index(drop=True)
tune_val   = tune_df.iloc[tune_val_idx].reset_index(drop=True)
print(f"Tuning set: train={len(tune_train)}, val={len(tune_val)}")

param_grid = [
    {'lr': 2e-5, 'batch_size': 16, 'epsilon': 0.3, 'weight_decay': 0.01},
    {'lr': 2e-5, 'batch_size': 32, 'epsilon': 0.5, 'weight_decay': 0.01},
    {'lr': 3e-5, 'batch_size': 32, 'epsilon': 0.5, 'weight_decay': 0.0},
    {'lr': 3e-5, 'batch_size': 16, 'epsilon': 0.3, 'weight_decay': 0.0},
    {'lr': 1e-5, 'batch_size': 16, 'epsilon': 0.5, 'weight_decay': 0.01},
    {'lr': 1e-5, 'batch_size': 32, 'epsilon': 0.3, 'weight_decay': 0.01}
][:3]

tuning_log = []

Tuning set: train=10147, val=2537


In [7]:
best_val_f1 = -1
best_config = None
best_model_state = None

for i, cfg in enumerate(param_grid):
    print(f"\n--- Tuning config {i+1}/{len(param_grid)}: {cfg} ---")
    
    train_dataset_tune = EmojiDataset(tune_train['text'].tolist(), tune_train['label_id'].tolist(), tokenizer)
    val_dataset_tune   = EmojiDataset(tune_val['text'].tolist(),   tune_val['label_id'].tolist(),   tokenizer)
    train_loader_tune = DataLoader(train_dataset_tune, batch_size=cfg['batch_size'], shuffle=True)
    val_loader_tune   = DataLoader(val_dataset_tune,   batch_size=cfg['batch_size'], shuffle=False)
    
    model = AutoModelForSequenceClassification.from_pretrained(
        "cardiffnlp/twitter-roberta-base-emoji-latest",
        num_labels=num_classes,
        ignore_mismatched_sizes=True
    ).to(device)
    
    if torch.cuda.device_count() >= 2:
        model = nn.DataParallel(model)
    
    optimizer = AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    fgm = FGM(model, epsilon=cfg['epsilon'])
    
    best_epoch_val_f1 = -1
    patience_counter = 0
    best_state_dict = None
    epoch_log = []
    
    for epoch in range(10):
        avg_loss = train_one_epoch(model, train_loader_tune, optimizer, criterion, device, fgm)
        preds, labels = evaluate(model, val_loader_tune, device)
        acc, f1 = compute_metrics(preds, labels)
        epoch_log.append({'epoch': epoch+1, 'train_loss': avg_loss, 'val_acc': acc, 'val_f1_macro': f1})
        print(f"Epoch {epoch+1}: loss={avg_loss:.4f}, val_acc={acc:.4f}, val_f1={f1:.4f}")
        
        if f1 > best_epoch_val_f1:
            best_epoch_val_f1 = f1
            patience_counter = 0
            if isinstance(model, nn.DataParallel):
                best_state_dict = model.module.state_dict()
            else:
                best_state_dict = model.state_dict()
        else:
            patience_counter += 1
            if patience_counter >= 3:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(best_state_dict)
    else:
        model.load_state_dict(best_state_dict)
    preds, labels = evaluate(model, val_loader_tune, device)
    _, final_f1 = compute_metrics(preds, labels)
    
    tuning_log.append({
        'config': cfg,
        'best_val_f1': final_f1,
        'epoch_log': epoch_log
    })
    
    if final_f1 > best_val_f1:
        best_val_f1 = final_f1
        best_config = cfg
        best_model_state = best_state_dict
    
    del model, optimizer, fgm
    torch.cuda.empty_cache()

print(f"\nBest config: {best_config} (val macro F1 = {best_val_f1:.4f})")


--- Tuning config 1/3: {'lr': 2e-05, 'batch_size': 16, 'epsilon': 0.3, 'weight_decay': 0.01} ---


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-emoji-latest
Key                        | Status   |                                                                                        
---------------------------+----------+----------------------------------------------------------------------------------------
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([100]) vs model:torch.Size([11])          
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([100, 768]) vs model:torch.Size([11, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Epoch 1: loss=1.9277, val_acc=0.4139, val_f1=0.2857
Epoch 2: loss=1.6025, val_acc=0.4206, val_f1=0.3360
Epoch 3: loss=1.3124, val_acc=0.4194, val_f1=0.3388
Epoch 4: loss=0.9850, val_acc=0.4147, val_f1=0.3285
Epoch 5: loss=0.6653, val_acc=0.3843, val_f1=0.3288
Epoch 6: loss=0.4231, val_acc=0.3788, val_f1=0.3196
Early stopping at epoch 6

--- Tuning config 2/3: {'lr': 2e-05, 'batch_size': 32, 'epsilon': 0.5, 'weight_decay': 0.01} ---


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-emoji-latest
Key                        | Status   |                                                                                        
---------------------------+----------+----------------------------------------------------------------------------------------
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([100]) vs model:torch.Size([11])          
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([100, 768]) vs model:torch.Size([11, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Epoch 1: loss=1.9523, val_acc=0.3946, val_f1=0.2527
Epoch 2: loss=1.6598, val_acc=0.4218, val_f1=0.3206
Epoch 3: loss=1.4296, val_acc=0.4202, val_f1=0.3291
Epoch 4: loss=1.1939, val_acc=0.4253, val_f1=0.3403
Epoch 5: loss=0.9303, val_acc=0.4028, val_f1=0.3259
Epoch 6: loss=0.6861, val_acc=0.4088, val_f1=0.3327
Epoch 7: loss=0.4710, val_acc=0.3863, val_f1=0.3249
Early stopping at epoch 7

--- Tuning config 3/3: {'lr': 3e-05, 'batch_size': 32, 'epsilon': 0.5, 'weight_decay': 0.0} ---


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-emoji-latest
Key                        | Status   |                                                                                        
---------------------------+----------+----------------------------------------------------------------------------------------
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([100]) vs model:torch.Size([11])          
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([100, 768]) vs model:torch.Size([11, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Epoch 1: loss=1.9444, val_acc=0.4147, val_f1=0.2964
Epoch 2: loss=1.6165, val_acc=0.4225, val_f1=0.3206
Epoch 3: loss=1.3448, val_acc=0.4166, val_f1=0.3373
Epoch 4: loss=1.0322, val_acc=0.4115, val_f1=0.3370
Epoch 5: loss=0.7225, val_acc=0.3843, val_f1=0.3349
Epoch 6: loss=0.4738, val_acc=0.4064, val_f1=0.3416
Epoch 7: loss=0.2829, val_acc=0.3918, val_f1=0.3240
Epoch 8: loss=0.1559, val_acc=0.3749, val_f1=0.3277
Epoch 9: loss=0.1040, val_acc=0.3749, val_f1=0.3212
Early stopping at epoch 9

Best config: {'lr': 2e-05, 'batch_size': 16, 'epsilon': 0.3, 'weight_decay': 0.01} (val macro F1 = 0.3304)


### 3.4 Final Training with Best Config on Full Data

In [8]:
full_train_dataset = EmojiDataset(train_df['text'].tolist(), train_df['label_id'].tolist(), tokenizer)
full_val_dataset   = EmojiDataset(val_df['text'].tolist(),   val_df['label_id'].tolist(),   tokenizer)
full_train_loader = DataLoader(full_train_dataset, batch_size=best_config['batch_size'], shuffle=True)
full_val_loader   = DataLoader(full_val_dataset,   batch_size=best_config['batch_size'], shuffle=False)

final_model = AutoModelForSequenceClassification.from_pretrained(
    "cardiffnlp/twitter-roberta-base-emoji-latest",
    num_labels=num_classes,
    ignore_mismatched_sizes=True
).to(device)
if torch.cuda.device_count() >= 2:
    final_model = nn.DataParallel(final_model)

optimizer = AdamW(final_model.parameters(), lr=best_config['lr'], weight_decay=best_config['weight_decay'])
criterion = nn.CrossEntropyLoss(weight=class_weights)
fgm = FGM(final_model, epsilon=best_config['epsilon'])

best_val_f1_full = -1
patience_counter = 0
best_state_full = None
full_training_log = []

for epoch in range(15):
    avg_loss = train_one_epoch(final_model, full_train_loader, optimizer, criterion, device, fgm)
    preds, labels = evaluate(final_model, full_val_loader, device)
    acc, f1 = compute_metrics(preds, labels)
    full_training_log.append({'epoch': epoch+1, 'train_loss': avg_loss, 'val_acc': acc, 'val_f1_macro': f1})
    print(f"Epoch {epoch+1}: loss={avg_loss:.4f}, val_acc={acc:.4f}, val_f1={f1:.4f}")
    
    if f1 > best_val_f1_full:
        best_val_f1_full = f1
        patience_counter = 0
        if isinstance(final_model, nn.DataParallel):
            best_state_full = final_model.module.state_dict()
        else:
            best_state_full = final_model.state_dict()
    else:
        patience_counter += 1
        if patience_counter >= 4:
            print("Early stopping triggered.")
            break

if isinstance(final_model, nn.DataParallel):
    final_model.module.load_state_dict(best_state_full)
else:
    final_model.load_state_dict(best_state_full)

test_dataset_full = EmojiDataset(test_df['text'].tolist(), test_df['label_id'].tolist(), tokenizer)
test_loader_full  = DataLoader(test_dataset_full, batch_size=best_config['batch_size'], shuffle=False)
test_preds, test_labels = evaluate(final_model, test_loader_full, device)
test_acc_fgm = accuracy_score(test_labels, test_preds)
test_f1_fgm  = f1_score(test_labels, test_preds, average='macro')
print(f"\nFGM Fine‑Tuned Transformer → Test Acc: {test_acc_fgm:.4f}, Macro F1: {test_f1_fgm:.4f}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-emoji-latest
Key                        | Status   |                                                                                        
---------------------------+----------+----------------------------------------------------------------------------------------
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([100]) vs model:torch.Size([11])          
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([100, 768]) vs model:torch.Size([11, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Epoch 1: loss=1.7712, val_acc=0.4787, val_f1=0.3548
Epoch 2: loss=1.5433, val_acc=0.4935, val_f1=0.3822
Epoch 3: loss=1.3388, val_acc=0.5238, val_f1=0.4448
Epoch 4: loss=1.1042, val_acc=0.5473, val_f1=0.4728
Epoch 5: loss=0.8686, val_acc=0.5510, val_f1=0.4987
Epoch 6: loss=0.6478, val_acc=0.5687, val_f1=0.5240
Epoch 7: loss=0.4672, val_acc=0.5707, val_f1=0.5301
Epoch 8: loss=0.3367, val_acc=0.5587, val_f1=0.5238
Epoch 9: loss=0.2493, val_acc=0.5681, val_f1=0.5328
Epoch 10: loss=0.1865, val_acc=0.5758, val_f1=0.5359
Epoch 11: loss=0.1449, val_acc=0.5689, val_f1=0.5345
Epoch 12: loss=0.1198, val_acc=0.5626, val_f1=0.5292
Epoch 13: loss=0.0987, val_acc=0.5786, val_f1=0.5442
Epoch 14: loss=0.0842, val_acc=0.5707, val_f1=0.5344
Epoch 15: loss=0.0716, val_acc=0.5774, val_f1=0.5410

FGM Fine‑Tuned Transformer → Test Acc: 0.6940, Macro F1: 0.6958


## 4. Save All Results

In [9]:
test_results = pd.DataFrame([
    {"Model": "TF‑IDF + LR", "Test Accuracy": test_acc_lr, "Test Macro F1": test_f1_lr},
    {"Model": "Baseline Transformer (no FT)", "Test Accuracy": test_acc_base, "Test Macro F1": test_f1_base},
    {"Model": "FGM Fine‑Tuned Transformer", "Test Accuracy": test_acc_fgm, "Test Macro F1": test_f1_fgm}
])
test_results.to_csv("/kaggle/working/test_results_all_models.csv", index=False)
print("Saved test_results_all_models.csv")

tuning_summary = []
for idx, rec in enumerate(tuning_log):
    tuning_summary.append({
        "config_index": idx+1,
        "learning_rate": rec['config']['lr'],
        "batch_size": rec['config']['batch_size'],
        "fgm_epsilon": rec['config']['epsilon'],
        "weight_decay": rec['config']['weight_decay'],
        "best_val_macro_f1": rec['best_val_f1']
    })
tuning_df = pd.DataFrame(tuning_summary)
tuning_df.to_csv("/kaggle/working/hp_tuning_results.csv", index=False)
print("Saved hp_tuning_results.csv")

for idx, rec in enumerate(tuning_log):
    log_df = pd.DataFrame(rec['epoch_log'])
    log_df.to_csv(f"/kaggle/working/tuning_config_{idx+1}_log.csv", index=False)
print("Saved per‑config tuning logs")

full_log_df = pd.DataFrame(full_training_log)
full_log_df.to_csv("/kaggle/working/final_full_training_log.csv", index=False)
print("Saved final_full_training_log.csv")

torch.save(best_state_full, "/kaggle/working/best_fgm_model.pt")
print("Saved best_fgm_model.pt")

Saved test_results_all_models.csv
Saved hp_tuning_results.csv
Saved per‑config tuning logs
Saved final_full_training_log.csv
Saved best_fgm_model.pt


## 5. Display Comparison
Just for quick review inside notebook

In [10]:
print("\n=== FINAL COMPARISON ===")
print(test_results.to_string(index=False))
print("\nBest hyperparameter config:", best_config)


=== FINAL COMPARISON ===
                       Model  Test Accuracy  Test Macro F1
                 TF‑IDF + LR       0.417063       0.332731
Baseline Transformer (no FT)       0.071032       0.045837
  FGM Fine‑Tuned Transformer       0.694048       0.695790

Best hyperparameter config: {'lr': 2e-05, 'batch_size': 16, 'epsilon': 0.3, 'weight_decay': 0.01}
